In [1]:
from votekit.cvr_loaders import load_scottish
from votekit.elections import STV, Borda
from votekit.utils import borda_scores, first_place_votes 

from glob import glob
from votekit.cvr_loaders import load_scottish
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import math, random

SCOT_ELEX_PATH = "/Users/cdonnay/Documents/GitHub/MGGG/scot-elex"

In [2]:
b_bloc_parties = ['Scottish National Party (SNP)', 'Green (Gr)']

In [4]:
def print_table(data, stat):

    round_to = 2
    for election_name, num_seats, bloc_to_cand_num, num_B_winners, fpv_share, borda_share, fpv_disprop, borda_disprop in data:

        election_name = election_name[:-4].split("_") 
        election_name = [x.capitalize() for x in election_name] 

        if "Ward" in election_name[-1]:
            ward_number = election_name[-1].split("d")[-1]
            election_name[-1] = f"Ward {ward_number}"
            
        election_name = " ".join(election_name)
        
        fpv_share = round(fpv_share, round_to)
        borda_share = round(borda_share, round_to)
        fpv_disprop  = round(fpv_disprop, round_to)
        borda_disprop = round(borda_disprop, round_to)


        if stat == "fpv":
            print(f"{election_name} & ${bloc_to_cand_num['A'], bloc_to_cand_num['B'], num_seats}$ & ${fpv_share:.2f}$ & ${round(fpv_share*num_seats, round_to):.2f}$ & {num_B_winners} & ${fpv_disprop:.2f}$ \\\\")
        elif stat == "borda":
            print(f"{election_name} & ${bloc_to_cand_num['A'], bloc_to_cand_num['B'], num_seats}$ & ${borda_share:.2f}$ & ${round(borda_share*num_seats, round_to):.2f}$ & {num_B_winners} & ${borda_disprop:.2f}$ \\\\")


## Scottish Elections Table

In [3]:
all_files = glob(f"{SCOT_ELEX_PATH}/*_cands/*.csv")

data = []

for file_name in all_files:
    scottish_profile, num_seats, cand_list, cand_to_party, ward_name = load_scottish(file_name)
    cand_to_bloc = {c:"B" if cand_to_party[c] in b_bloc_parties 
                else "A" for c in cand_list}

    bloc_to_cand_num = {"A": len([c for c, bloc in cand_to_bloc.items() if bloc == "A"]),
                    "B": len([c for c, bloc in cand_to_bloc.items() if bloc == "B"])}

    fpv_dict = first_place_votes(scottish_profile, to_float=True)
    fpv_share = sum(v for c,v in fpv_dict.items() if cand_to_bloc[c]=="B")/sum(fpv_dict.values())

    if bloc_to_cand_num["B"] < math.floor(fpv_share*num_seats) or bloc_to_cand_num["B"] == 0:
        continue 

    borda_dict = borda_scores(scottish_profile, borda_max=num_seats, to_float=True)
    borda_share = sum(v for c,v in borda_dict.items() if cand_to_bloc[c]=="B")/sum(borda_dict.values())

    # record number of B winners
    election  =STV(profile = scottish_profile, m = num_seats)
    winners= [c for s in election.get_elected() for c in s]
    num_B_winners = len([c for c in winners if cand_to_bloc[c] == "B"])
    b_winner_share = num_B_winners/num_seats

    fpv_disprop = b_winner_share - fpv_share
    borda_disprop = b_winner_share - borda_share

    data.append((file_name.split('/')[-1], num_seats, bloc_to_cand_num, num_B_winners, fpv_share, borda_share, round(fpv_disprop, 4), round(borda_disprop, 4)))
    

## Split by number of seats up for election

In [5]:
three_seats = [x for x in data if x[1]==3]
four_seats = [x for x in data if x[1]==4]
five_seats = [x for x in data if x[1]==5]

In [6]:
sample_size = 20
random.seed(47)

three_seat_sample = random.sample(three_seats, sample_size)
four_seat_sample = random.sample(four_seats, sample_size)


### FPV

In [7]:
three_seat_sample_sorted = sorted(three_seat_sample, key = lambda x: x[4]) # 4 is fpv, 5 is borda
four_seat_sample_sorted = sorted(four_seat_sample, key = lambda x: x[4])
five_seats_sorted = sorted(five_seats, key = lambda x: x[4])

In [8]:
print_table(three_seat_sample_sorted, "fpv")
print_table(four_seat_sample_sorted, "fpv")
print_table(five_seats_sorted, "fpv")

Sc Borders 2017 Ward 10 & $(4, 2, 3)$ & $0.15$ & $0.45$ & 1 & $0.19$ \\
Aberdeen 2017 Ward 9 & $(6, 2, 3)$ & $0.18$ & $0.54$ & 0 & $-0.18$ \\
Sc Borders 2012 Ward 4 & $(5, 1, 3)$ & $0.20$ & $0.60$ & 0 & $-0.20$ \\
Inverclyde 2012 Ward 2 & $(5, 1, 3)$ & $0.22$ & $0.66$ & 1 & $0.11$ \\
Sc Borders 2017 Ward 7 & $(5, 2, 3)$ & $0.23$ & $0.69$ & 1 & $0.10$ \\
South Ayrshire 2022 Ward 8 & $(6, 1, 3)$ & $0.25$ & $0.75$ & 1 & $0.08$ \\
Sc Borders 2017 Ward 6 & $(4, 2, 3)$ & $0.25$ & $0.75$ & 1 & $0.08$ \\
Aberdeenshire 2017 Ward 1 & $(3, 1, 3)$ & $0.25$ & $0.75$ & 1 & $0.08$ \\
Highland 2017 Ward 15 & $(5, 1, 3)$ & $0.26$ & $0.78$ & 1 & $0.07$ \\
Perth Kinross 2017 Ward 6 & $(5, 3, 3)$ & $0.28$ & $0.84$ & 1 & $0.05$ \\
Highland 2012 Ward 14 & $(6, 2, 3)$ & $0.30$ & $0.90$ & 1 & $0.03$ \\
South Lanarkshire 2022 Ward 9 & $(6, 2, 3)$ & $0.31$ & $0.93$ & 1 & $0.02$ \\
Aberdeen 2022 Ward 3 & $(4, 3, 3)$ & $0.36$ & $1.08$ & 1 & $-0.02$ \\
Renfrewshire 2017 Ward 3 & $(6, 3, 3)$ & $0.40$ & $1.20$ & 1 &

### Borda

In [9]:
three_seat_sample_sorted = sorted(three_seat_sample, key = lambda x: x[5]) # 4 is fpv, 5 is borda
four_seat_sample_sorted = sorted(four_seat_sample, key = lambda x: x[5])
five_seats_sorted = sorted(five_seats, key = lambda x: x[5])

In [10]:
print_table(three_seat_sample_sorted, "borda")
print_table(four_seat_sample_sorted, "borda")
print_table(five_seats_sorted, "borda")

Sc Borders 2012 Ward 4 & $(5, 1, 3)$ & $0.17$ & $0.51$ & 0 & $-0.17$ \\
Inverclyde 2012 Ward 2 & $(5, 1, 3)$ & $0.17$ & $0.51$ & 1 & $0.16$ \\
Aberdeen 2017 Ward 9 & $(6, 2, 3)$ & $0.18$ & $0.54$ & 0 & $-0.18$ \\
South Ayrshire 2022 Ward 8 & $(6, 1, 3)$ & $0.19$ & $0.57$ & 1 & $0.14$ \\
Highland 2017 Ward 15 & $(5, 1, 3)$ & $0.19$ & $0.57$ & 1 & $0.14$ \\
Sc Borders 2017 Ward 10 & $(4, 2, 3)$ & $0.19$ & $0.57$ & 1 & $0.14$ \\
Aberdeenshire 2017 Ward 1 & $(3, 1, 3)$ & $0.20$ & $0.60$ & 1 & $0.13$ \\
Sc Borders 2017 Ward 7 & $(5, 2, 3)$ & $0.21$ & $0.63$ & 1 & $0.12$ \\
Sc Borders 2017 Ward 6 & $(4, 2, 3)$ & $0.24$ & $0.72$ & 1 & $0.09$ \\
Perth Kinross 2017 Ward 6 & $(5, 3, 3)$ & $0.30$ & $0.90$ & 1 & $0.03$ \\
South Lanarkshire 2022 Ward 9 & $(6, 2, 3)$ & $0.33$ & $0.99$ & 1 & $-0.00$ \\
Highland 2012 Ward 14 & $(6, 2, 3)$ & $0.34$ & $1.02$ & 1 & $-0.01$ \\
East Dunbartonshire 2022 Ward 6 & $(4, 2, 3)$ & $0.38$ & $1.14$ & 1 & $-0.05$ \\
Argyll Bute 2022 Ward 6 & $(5, 2, 3)$ & $0.40$ & 

## Split by number of B winners

In [11]:
zero_winners = [x for x in data if x[3]==0]
one_winners = [x for x in data if x[3]==1]
two_winners = [x for x in data if x[ 3]==2]
three_winners = [x for x in data if x[3]==3]

In [13]:
random.seed(47)

zero_winners_sample = random.sample(zero_winners, 5)
one_winners_sample = random.sample(one_winners, 10)
two_winners_sample = random.sample(two_winners, 10)
three_winners_sample = random.sample(three_winners, 5)



### FPV

In [14]:
sorted_list = zero_winners_sample+one_winners_sample+two_winners_sample+three_winners_sample
sorted_list = sorted(sorted_list, key = lambda x: x[1]*x[4]) # FPV seat share 
print_table(sorted_list, "fpv")

North Ayrshire 2022 Arran & $(4, 2, 1)$ & $0.36$ & $0.36$ & 0 & $-0.36$ \\
Orkney 2022 Ward 3 & $(5, 1, 3)$ & $0.14$ & $0.42$ & 0 & $-0.14$ \\
Highland 2012 Ward 12 & $(7, 1, 3)$ & $0.15$ & $0.45$ & 0 & $-0.15$ \\
Dumgal 2022 Ward 12 & $(5, 2, 3)$ & $0.17$ & $0.51$ & 0 & $-0.17$ \\
Inverclyde 2017 Ward 5 & $(6, 1, 3)$ & $0.24$ & $0.72$ & 1 & $0.09$ \\
Edinburgh 2017 Ward 8 & $(4, 2, 3)$ & $0.25$ & $0.75$ & 0 & $-0.25$ \\
Eilean Siar 2012 Ward 4 & $(5, 2, 3)$ & $0.32$ & $0.96$ & 1 & $0.01$ \\
Fife 2012 Ward 12 & $(4, 2, 3)$ & $0.32$ & $0.96$ & 1 & $0.01$ \\
North Lanarkshire 2012 Ward 19 & $(4, 2, 4)$ & $0.26$ & $1.04$ & 1 & $-0.01$ \\
Clackmannanshire 2022 Ward 5 & $(5, 2, 3)$ & $0.37$ & $1.11$ & 1 & $-0.04$ \\
Inverclyde 2017 Ward 7 & $(6, 2, 3)$ & $0.38$ & $1.14$ & 1 & $-0.05$ \\
Stirling 2017 Ward 5 & $(3, 3, 3)$ & $0.38$ & $1.14$ & 1 & $-0.05$ \\
East Renfrewshire 2022 Ward 5 & $(7, 2, 4)$ & $0.29$ & $1.16$ & 1 & $-0.04$ \\
North Lanarkshire 2017 Ward 8 & $(5, 3, 4)$ & $0.30$ & $1.

### Borda

In [16]:
sorted_list = zero_winners_sample+one_winners_sample+two_winners_sample+three_winners_sample
sorted_list = sorted(sorted_list, key = lambda x: x[1]*x[5]) # Borda seat share 
print_table(sorted_list, "borda")

Orkney 2022 Ward 3 & $(5, 1, 3)$ & $0.11$ & $0.33$ & 0 & $-0.11$ \\
North Ayrshire 2022 Arran & $(4, 2, 1)$ & $0.36$ & $0.36$ & 0 & $-0.36$ \\
Highland 2012 Ward 12 & $(7, 1, 3)$ & $0.12$ & $0.36$ & 0 & $-0.12$ \\
Inverclyde 2017 Ward 5 & $(6, 1, 3)$ & $0.17$ & $0.51$ & 1 & $0.16$ \\
Dumgal 2022 Ward 12 & $(5, 2, 3)$ & $0.17$ & $0.51$ & 0 & $-0.17$ \\
Edinburgh 2017 Ward 8 & $(4, 2, 3)$ & $0.24$ & $0.72$ & 0 & $-0.24$ \\
Eilean Siar 2012 Ward 4 & $(5, 2, 3)$ & $0.31$ & $0.93$ & 1 & $0.02$ \\
Clackmannanshire 2022 Ward 5 & $(5, 2, 3)$ & $0.33$ & $0.99$ & 1 & $0.01$ \\
Fife 2012 Ward 12 & $(4, 2, 3)$ & $0.34$ & $1.02$ & 1 & $-0.00$ \\
East Renfrewshire 2022 Ward 5 & $(7, 2, 4)$ & $0.25$ & $1.00$ & 1 & $-0.00$ \\
North Lanarkshire 2012 Ward 19 & $(4, 2, 4)$ & $0.29$ & $1.16$ & 1 & $-0.04$ \\
Inverclyde 2017 Ward 7 & $(6, 2, 3)$ & $0.40$ & $1.20$ & 1 & $-0.06$ \\
Glasgow 2007 Hillhead & $(8, 2, 4)$ & $0.33$ & $1.32$ & 2 & $0.17$ \\
Stirling 2017 Ward 5 & $(3, 3, 3)$ & $0.45$ & $1.35$ & 1 &